# Ironhack Project: Strategic Analysis Report
## Phase 2: SQL Queries & Python Visualization
This notebook contains the data story, SQL analysis, and visualizations used in the final presentation.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('dark')

### Database Setup (SQLite for local analysis)
Loading the clean CSVs into a local SQL database to run our queries.

In [2]:
# Create in-memory SQLite database
conn = sqlite3.connect(':memory:')

# Load CSVs to SQL
pd.read_csv('products.csv').to_sql('products', conn, index=False)
pd.read_csv('salespersons.csv').to_sql('salespersons', conn, index=False)
pd.read_csv('invoices.csv').to_sql('orders', conn, index=False)

print('Database initialized.')

FileNotFoundError: [Errno 2] No such file or directory: 'products.csv'

### Question 1: Which country is the best market to prioritize?
We use SQL to aggregate total sales and marketing spend by country to calculate the ROI.

In [ ]:
query1 = """
SELECT 
    Country, 
    SUM(Amount) as Total_Sales,
    SUM(Marketing_Spend) as Total_Marketing,
    (SUM(Amount) / SUM(Marketing_Spend)) as ROI
FROM orders
GROUP BY Country
ORDER BY Total_Sales DESC;
"""

country_stats = pd.read_sql(query1, conn)
display(country_stats)

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=country_stats, x='Country', y='Total_Sales', color='#8B0000')
plt.title('Total Sales by Country ($)', fontsize=16, fontweight='bold')
plt.ylabel('Total Sales', fontsize=12)
plt.xlabel('Country', fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

**Insight**: Australia is the top performing market, generating over $1.3B with the highest Marketing ROI (9.54).

### Question 2: Which products and channels generate the strongest demand?
Using SQL JOINs to analyze product performance.

In [ ]:
query2 = """
SELECT 
    p.Product,
    o.Channel,
    SUM(o.Amount) as Total_Sales
FROM orders o
JOIN products p ON o.Product_ID = p.Product_ID
GROUP BY p.Product, o.Channel
ORDER BY Total_Sales DESC
LIMIT 10;
"""

product_channel = pd.read_sql(query2, conn)
display(product_channel)

In [ ]:
channel_totals = pd.read_sql("SELECT Channel, SUM(Amount) as Total FROM orders GROUP BY Channel", conn)

plt.figure(figsize=(8, 8))
plt.pie(channel_totals['Total'], labels=channel_totals['Channel'], autopct='%1.1f%%', 
        colors=['#8B0000', '#1A0B08', '#D3D3D3'], textprops={'color':"w", 'weight':'bold'})
plt.title('Sales Distribution by Channel', fontsize=16, fontweight='bold')
plt.show()

**Insight**: The 85% Dark Bar is our top product, and Wholesale is the dominant channel, driving the majority of our volume.

### Question 3: How does the sales team perform?
Analyzing total sales and average discount rates per salesperson.

In [ ]:
query3 = """
SELECT 
    s.Salesperson,
    SUM(o.Amount) as Total_Sales,
    AVG(o.Discount_Pct) * 100 as Avg_Discount_Percent
FROM orders o
JOIN salespersons s ON o.Salesperson_ID = s.Salesperson_ID
GROUP BY s.Salesperson
ORDER BY Total_Sales DESC;
"""

sales_perf = pd.read_sql(query3, conn)
display(sales_perf)

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=sales_perf, y='Salesperson', x='Total_Sales', color='#1A0B08')
plt.title('Total Sales by Salesperson ($)', fontsize=16, fontweight='bold')
plt.xlabel('Total Sales', fontsize=12)
plt.ylabel('')
plt.tight_layout()
plt.show()

**Insight**: Ananya Gupta leads the team. There is a strong correlation between higher average discounts (~15.1%) and higher total sales volume, indicating discounts are effectively used to close large wholesale deals.

In [ ]:

# Set the visual style (to match the presentation aesthetic)
plt.style.use('seaborn-v0_8-whitegrid')

# Create the bar chart
plt.figure(figsize=(10, 6))
sns.barplot(data=country_stats, x='Country', y='Total_Sales', color='#8B0000')

# Customization and labels
plt.title('Total Sales by Country ($)', fontsize=16, fontweight='bold')
plt.ylabel('Total Sales', fontsize=12)
plt.xlabel('Country', fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()

# Display the chart
plt.show()


1. Chart: Sales Distribution by Channel (Pie Chart)
We generate the pie chart showing the dominance of the Wholesale channel, using the presentation's color palette (Dark Red, Dark Brown/Black, Light Grey).

In [ ]:
# Create the pie chart
plt.figure(figsize=(8, 8))
plt.pie(
    channel_totals['Total'], 
    labels=channel_totals['Channel'], 
    autopct='%1.1f%%', 
    colors=['#8B0000', '#1A0B08', '#D3D3D3'], 
    textprops={'color': "w", 'weight': 'bold'}
)

# Customization and labels
plt.title('Sales Distribution by Channel', fontsize=16, fontweight='bold')

# Display the chart
plt.show()


2. Chart: Salesperson Performance (Horizontal Bar Chart)
We generate the black horizontal bar chart ranking the sales team, with Ananya Gupta at the top.

In [ ]:
# Create the horizontal bar chart
plt.figure(figsize=(10, 6))
sns.barplot(data=sales_perf, y='Salesperson', x='Total_Sales', color='#1A0B08')

# Customization and labels
plt.title('Total Sales by Salesperson ($)', fontsize=16, fontweight='bold')
plt.xlabel('Total Sales', fontsize=12)
plt.ylabel('') # Hide the Y-axis label since the names are self-explanatory
plt.tight_layout()

# Display the chart
plt.show()
